# A Hands-On Introduction to Deep Learning for Computer Vision

## BMVA Summer School — Practical Session (~1 hour)

<a href="https://colab.research.google.com/github/YOUR-GITHUB-USERNAME/YOUR-REPO/blob/main/BMVA_MNIST_CNN_Practical.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


---

Welcome! In this practical you will build, train and test your first **convolutional neural network (CNN)** — the family of models behind most of modern computer vision — using [PyTorch](https://pytorch.org/).

Our task is a classic: teaching a network to recognise **handwritten digits** (0–9) from the **MNIST** dataset. It is small enough to train in a couple of minutes, yet rich enough to demonstrate every ingredient of real deep learning systems.

### What you will learn

1. **Data** — how images become tensors, and how PyTorch loads them in batches
2. **Convolutions** — what they compute and why they suit images
3. **Models** — how to define a CNN with `torch.nn`
4. **Training** — *everything* you need to train a network: a loss function, an optimiser, and the training loop
5. **Evaluation** — accuracy, confusion matrices, and looking at the mistakes
6. **Exercises** — build a deeper network and experiment with different loss functions

### How to use this notebook

- Run each cell in order with **Shift + Enter**.
- **🧠 Quick check** boxes ask you a short question about what you have just seen — think first, then click to reveal the answer.
- **✏️ Exercises** in Part 6 are where you write code yourself. Solutions are at the very end, but do try first!

### Suggested timing

| Part | Topic | Time |
|---|---|---|
| 0 | Setup | 5 min |
| 1 | The data: MNIST | 8 min |
| 2 | Convolutions | 8 min |
| 3 | Building a CNN | 7 min |
| 4 | Training | 12 min |
| 5 | Evaluation | 7 min |
| 6 | Exercises | 15 min |

(Exercise 3 is an optional extension — if the hour runs out, it makes a good take-home.)

This notebook is self-contained and needs nothing beyond `torch`, `torchvision`, `matplotlib` and `numpy` — all pre-installed on Google Colab. It runs in a few minutes even without a GPU.

> *Inspired by the [dl-pytorch](https://github.com/atapour/dl-pytorch) teaching materials by Amir Atapour-Abarghouei (Durham University).*

---
# Part 0 — Setup

**On Google Colab:** a GPU makes training faster, but is *not required* for this practical. To use one, go to **Runtime → Change runtime type → Hardware accelerator → T4 GPU** *before* running any cells.

**Running locally instead?** Install the dependencies with:
`pip install torch torchvision matplotlib numpy`

Let's start by importing what we need:

In [ ]:
import torch                                  # the core library: tensors and automatic differentiation
import torch.nn as nn                         # neural network layers and loss functions
import torch.nn.functional as F               # the same operations as plain functions
from torch.utils.data import DataLoader       # batching and shuffling of datasets

import torchvision
from torchvision import datasets, transforms  # standard vision datasets and image transforms

import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version:     {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

PyTorch code is *device-agnostic*: the same code runs on a CPU, an NVIDIA GPU (`cuda`) or an Apple Silicon GPU (`mps`). We pick the best device available and later move our data and model onto it.

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")     # NVIDIA GPU (this is what you get on Colab)
elif torch.backends.mps.is_available():
    device = torch.device("mps")      # Apple Silicon GPU
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# Fix the random seed so that everyone gets (roughly) the same results:
torch.manual_seed(42);

---
# Part 1 — The data: MNIST

Deep learning starts with data. **MNIST** contains 70,000 grayscale images of handwritten digits, each **28 × 28 pixels**, split into 60,000 training images and 10,000 test images. Every image comes with a **label**: the digit it shows (0–9).

Why two separate sets?

- The **training set** is what the network learns from.
- The **test set** is kept aside to measure how well the network handles digits it has *never seen*. Testing on training data would be like marking an exam whose questions the student memorised in advance.

`torchvision` downloads MNIST for us. We attach a small pipeline of **transforms** that is applied to every image as it is loaded:

1. `ToTensor()` converts the image to a PyTorch **tensor** (a multi-dimensional array) of shape `[1, 28, 28]` — that is `[channels, height, width]`, with 1 channel because the images are grayscale — and scales pixel values from 0–255 down to 0–1.
2. `Normalize(mean, std)` then standardises the values using MNIST's known mean (0.1307) and standard deviation (0.3081). Inputs centred around zero make training smoother and faster.

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),                        # image -> tensor [1, 28, 28], values in [0, 1]
    transforms.Normalize((0.1307,), (0.3081,)),   # standardise with MNIST's mean and std
])

train_dataset = datasets.MNIST(root="./data", train=True,  download=True, transform=transform)
test_dataset  = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples:     {len(test_dataset)}")

A dataset behaves like a list: indexing it gives one `(image, label)` pair. Let's inspect a single sample:

In [ ]:
image, label = train_dataset[0]

print(f"Image type:  {type(image).__name__}")
print(f"Image shape: {list(image.shape)}   # [channels, height, width]")
print(f"Pixel range: {image.min():.2f} to {image.max():.2f}   # after normalisation")
print(f"Label:       {label}")

Networks are not trained one image at a time but on **mini-batches** — here, 128 images at once. Batches average the gradient over many examples (a more reliable learning signal) and make much better use of parallel hardware.

The **`DataLoader`** handles batching for us, and also **shuffles** the training data so that each **epoch** — each full pass over the training set — presents the images in a fresh random order.

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False)

# Grab one batch to check the shapes:
images, labels = next(iter(train_loader))
print(f"One batch of images: {list(images.shape)}   # [batch, channels, height, width]")
print(f"One batch of labels: {list(labels.shape)}")

Always **look at your data** — many deep learning bugs are really data bugs:

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(12, 3.5))
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i, 0], cmap="gray")   # images[i, 0]: i-th image, first (only) channel
    ax.set_title(f"label: {labels[i].item()}")
    ax.axis("off")
plt.tight_layout()
plt.show()

---
### 🧠 Quick check 1

**Q:** Why did we set `shuffle=True` for the training loader but `shuffle=False` for the test loader?

<details><summary><b>Show answer</b></summary>

**Training:** the dataset is stored in an arbitrary order, and without shuffling the network would see the images in exactly the same sequence every epoch. Any ordering pattern (e.g. long runs of the same digit) biases the gradient updates; fresh random batches give more representative gradients and better training.

**Testing:** we only *measure* performance — no weights are updated — so the order cannot change the result. Keeping it fixed also makes evaluation reproducible.

</details>

---

---
# Part 2 — Convolutions: the building block of vision models

Could we not just feed the 784 pixels (28 × 28) straight into ordinary fully-connected layers? We could — but two things go wrong:

1. **It ignores spatial structure.** Flattening treats pixel (3, 4) and pixel (4, 4) as unrelated inputs, throwing away the very thing that makes an image an image.
2. **It doesn't scale.** A fully-connected layer from 784 pixels to just 128 units already needs ~100,000 weights. For a 224 × 224 colour photo, the same layer would need ~19 *million*.

A **convolutional layer** fixes both. It slides a small filter (also called a *kernel*), e.g. **3 × 3 weights**, across the image. At each position it computes a weighted sum of the pixels under the filter, producing one output pixel. The result is a **feature map** showing *where* in the image the pattern encoded by the filter occurs.

Two ideas make this powerful:

- **Local connectivity** — each output looks at a small neighbourhood, respecting spatial structure.
- **Weight sharing** — the *same* 9 weights are used at every position, so a pattern (an edge, a corner, a pen-stroke) is detected wherever it appears, and the layer needs only a handful of parameters.

Before CNNs, computer vision researchers designed such filters *by hand*. Here is a classic — the **Sobel filter**, which responds to vertical edges. Let's apply it to a digit with `F.conv2d` and see what it does:

In [ ]:
image, label = train_dataset[1]   # one digit image, shape [1, 28, 28]

# A hand-crafted 3x3 filter that responds to vertical edges.
# Shape [1, 1, 3, 3] = [output channels, input channels, height, width]:
sobel = torch.tensor([[-1., 0., 1.],
                      [-2., 0., 2.],
                      [-1., 0., 1.]]).reshape(1, 1, 3, 3)

with torch.no_grad():
    edges = F.conv2d(image.unsqueeze(0), sobel, padding=1)   # unsqueeze adds a batch dimension

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
axes[0].imshow(image[0], cmap="gray")
axes[0].set_title(f"input (a {label})")
axes[1].imshow(sobel[0, 0], cmap="gray")
axes[1].set_title("3x3 filter")
axes[2].imshow(edges[0, 0], cmap="gray")
axes[2].set_title("output: vertical edges")
for ax in axes:
    ax.axis("off")
plt.show()

**The key idea of deep learning:** nobody designs the filters any more. The filter values are the network's **weights**, and training *learns* them from data. Early layers typically discover edge- and blob-detectors much like Sobel; deeper layers combine them into detectors for curves, loops, and eventually whole digits.

Two more ingredients complete the picture:

- **ReLU** (`max(0, x)`) — a simple non-linearity applied after each layer. Without it, a stack of convolutions would collapse mathematically into one single linear operation, no matter how deep. *(Try to convince yourself why!)*
- **Max-pooling** (`nn.MaxPool2d(2)`) — keeps the maximum of each 2 × 2 block, halving the width and height. This shrinks the computation and makes the features robust to small shifts of the input.

**Shape arithmetic** you will use constantly. Two knobs control a convolution's geometry: **padding** adds a border of $p$ zero-valued pixels around the input (so the filter can sit centred on the edge pixels), and **stride** is how many pixels the filter jumps between positions ($s = 1$ everywhere in this notebook). A convolution with kernel size $k$, padding $p$ and stride $s$ maps an input of size $H$ to output size

$$H_{out} = \left\lfloor \frac{H + 2p - k}{s} \right\rfloor + 1$$

With $k=3, p=1, s=1$ (our usual setting) the size is **unchanged**; a 2 × 2 max-pool then **halves** it.

---
### 🧠 Quick check 2

**Q1:** A 28 × 28 image goes through a 3 × 3 convolution with `padding=1`, then a 2 × 2 max-pool. What is the output size?

**Q2:** How many learnable parameters does `nn.Conv2d(1, 16, kernel_size=3)` have? (Don't forget that each of the 16 filters also has a bias term.)

<details><summary><b>Show answer</b></summary>

**Q1:** The convolution keeps 28 × 28 (padding 1 exactly compensates the 3 × 3 kernel: $(28 + 2 - 3) + 1 = 28$); the pool then halves it to **14 × 14**.

**Q2:** Each filter has $1 \times 3 \times 3 = 9$ weights plus 1 bias, and there are 16 filters: $16 \times (9 + 1) = $ **160 parameters**. Compare that with the ~100,000 of a small fully-connected layer!

</details>

---

---
# Part 3 — Building a CNN

In PyTorch, a model is a class that inherits from **`nn.Module`** and defines two things:

- **`__init__`** — creates the layers (which hold the learnable weights);
- **`forward`** — says how data flows through those layers.

Our network stacks two *conv → ReLU → pool* blocks, then flattens the feature maps into a vector and applies two fully-connected (`Linear`) layers to produce **10 scores, one per digit**:

```text
input                [B,  1, 28, 28]
Conv2d(1→16) + ReLU  [B, 16, 28, 28]     3x3 kernels, padding 1
MaxPool(2)           [B, 16, 14, 14]
Conv2d(16→32) + ReLU [B, 32, 14, 14]     3x3 kernels, padding 1
MaxPool(2)           [B, 32,  7,  7]
flatten              [B, 1568]           32 x 7 x 7 = 1568
Linear(1568→128) + ReLU
Linear(128→10)       [B, 10]             one score per digit class
```

The 10 outputs are raw, unnormalised scores called **logits** — they can be any real number, and bigger means "more likely this class". We deliberately do *not* apply softmax inside the model; you will see why in Part 4.

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)   # 1 input channel  -> 16 feature maps
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)  # 16 feature maps  -> 32 feature maps
        self.pool  = nn.MaxPool2d(2)                               # halves height and width
        self.fc1   = nn.Linear(32 * 7 * 7, 128)
        self.fc2   = nn.Linear(128, 10)                            # 10 output classes

    def forward(self, x):                        # x: [B, 1, 28, 28]
        x = self.pool(F.relu(self.conv1(x)))     # -> [B, 16, 14, 14]
        x = self.pool(F.relu(self.conv2(x)))     # -> [B, 32, 7, 7]
        x = torch.flatten(x, start_dim=1)        # -> [B, 1568]  (keep the batch dimension!)
        x = F.relu(self.fc1(x))                  # -> [B, 128]
        return self.fc2(x)                       # -> [B, 10] logits

model = SimpleCNN().to(device)   # build it and move the weights to our device
print(model)

Before training anything, always **sanity-check the model**: push a dummy batch through and confirm the output shape, and count the parameters we are about to train.

In [ ]:
dummy = torch.randn(2, 1, 28, 28, device=device)   # a fake batch of 2 "images"
out = model(dummy)
print(f"Output shape: {list(out.shape)}   # [batch, classes] — looks right!")

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {n_params:,}")

---
### 🧠 Quick check 3

**Q:** Why is the input size of `fc1` exactly `32 * 7 * 7`? And where does most of this network's ~207k parameters live?

<details><summary><b>Show answer</b></summary>

After two poolings the 28 × 28 image has shrunk to 7 × 7 (28 → 14 → 7), and the second conv layer produces 32 channels, so flattening gives a vector of $32 \times 7 \times 7 = 1568$ numbers — which is what `fc1` must accept.

The parameters are dominated by `fc1`: $1568 \times 128 + 128 = 200{,}832$ of the total 206,922 — about 97%! The two conv layers contribute only 4,800 parameters between them. This is typical: convolutions are very parameter-efficient, fully-connected layers are not.

</details>

---

---
# Part 4 — Training

Right now the network's weights are random numbers, so its predictions are guesses. **Training** means adjusting the weights, little by little, to make predictions match the labels on the training set. Beyond the data and model we already have, this needs exactly **three more things**:

### 1. A loss function

A single number measuring *how wrong* the predictions are — lower is better. Training is nothing more than minimising this number.

For classification, the standard choice is **cross-entropy loss** (`nn.CrossEntropyLoss`). Given the 10 logits, it (internally) applies **softmax** to turn them into probabilities that sum to 1, then takes the **negative log** of the probability assigned to the *correct* class. Confidently right → loss near 0; confidently wrong → large loss. This is also why our model outputs raw logits: `nn.CrossEntropyLoss` *expects logits* and does the softmax itself, in a numerically stable way.

PyTorch ships a whole catalogue of loss functions for different tasks — see the [`torch.nn` docs](https://docs.pytorch.org/docs/stable/nn.html#loss-functions). You will experiment with others in Exercise 2.

### 2. An optimiser

The rule for updating the weights. Backpropagation (`loss.backward()`) gives us the **gradient** of the loss with respect to every weight — the direction that would *increase* the loss — so the optimiser steps the other way:

$$w \leftarrow w - \eta \cdot \nabla_w \, \mathcal{L}$$

where $\eta$ is the **learning rate**, the most important hyperparameter in deep learning. Plain gradient descent is `optim.SGD`; we use **`optim.Adam`**, a variant that adapts the step size per weight and usually "just works" with $\eta = 10^{-3}$. Other choices live in [`torch.optim`](https://docs.pytorch.org/docs/stable/optim.html).

### 3. The training loop

Deep learning frameworks do *not* hide the loop from you — writing it yourself is normal PyTorch practice, and it is always the same five steps:

1. take a batch of data, move it to the device
2. **forward pass** — run the model to get predictions
3. compute the **loss**
4. **backward pass** — `loss.backward()` computes all gradients (after zeroing the old ones!)
5. **update** — `optimiser.step()` adjusts the weights

One full pass over the training set is called an **epoch**.

In [ ]:
criterion = nn.CrossEntropyLoss()
optimiser = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def train_one_epoch(model, loader, criterion, optimiser):
    # One full pass over the training data. Returns the per-batch losses.
    model.train()                    # put the model in training mode
    batch_losses = []

    for batch_idx, (images, labels) in enumerate(loader):
        images = images.to(device)                  # step 1: batch onto the device
        labels = labels.to(device)

        logits = model(images)                      # step 2: forward pass
        loss = criterion(logits, labels)            # step 3: how wrong are we?

        optimiser.zero_grad()                       # step 4: clear old gradients...
        loss.backward()                             #         ...and compute new ones
        optimiser.step()                            # step 5: update the weights

        batch_losses.append(loss.item())            # .item(): tensor -> plain float, for logging
        if batch_idx % 100 == 0:
            print(f"   batch {batch_idx:3d}/{len(loader)}   loss: {loss.item():.4f}")

    return batch_losses

We also want to measure **accuracy**: the fraction of images classified correctly. The predicted class is simply the position of the *largest* logit (`argmax`). Note the two evaluation habits — `model.eval()` and `torch.no_grad()` — explained in the quick check below.

In [ ]:
def accuracy(model, loader):
    # Fraction of samples classified correctly.
    model.eval()                     # evaluation mode
    correct, total = 0, 0
    with torch.no_grad():            # we are only measuring — no gradients needed
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)
            logits = model(images)
            predictions = logits.argmax(dim=1)          # index of the largest logit = predicted class
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    return correct / total

Time to train — **2 epochs**, which takes roughly half a minute on a Colab GPU and a couple of minutes on a CPU. Before we start, check the untrained accuracy: with 10 classes, random guessing should give about 10%. Watch the loss fall as the network learns.

In [ ]:
print(f"Test accuracy BEFORE training: {accuracy(model, test_loader):.1%}   (random guessing ≈ 10%)\n")

n_epochs = 2
all_losses = []

for epoch in range(n_epochs):
    print(f"Epoch {epoch + 1}/{n_epochs}")
    all_losses += train_one_epoch(model, train_loader, criterion, optimiser)
    print(f"   test accuracy: {accuracy(model, test_loader):.1%}\n")

From 10% to ~98–99% in two epochs. Let's plot the loss over the training steps — the raw per-batch loss is noisy (every batch is different), so we overlay a moving average to see the trend:

In [ ]:
plt.figure(figsize=(9, 3.5))
plt.plot(all_losses, alpha=0.3, label="per-batch loss")

window = 25
smoothed = np.convolve(all_losses, np.ones(window) / window, mode="valid")
plt.plot(np.arange(window - 1, len(all_losses)), smoothed, label=f"moving average ({window} batches)")

plt.xlabel("training step (batch)")
plt.ylabel("cross-entropy loss")
plt.legend()
plt.show()

---
### 🧠 Quick check 4

**Q1:** What would happen if we forgot `optimiser.zero_grad()` in the loop?

**Q2:** Why do we call `model.eval()` and wrap evaluation in `torch.no_grad()`?

<details><summary><b>Show answer</b></summary>

**Q1:** PyTorch *accumulates* gradients into `.grad` on every `backward()` call. Without zeroing, each step would apply the sum of all gradients so far — the updates grow and point in stale directions, and training typically diverges. (Forgetting `zero_grad()` is one of the most common PyTorch bugs!)

**Q2:** `model.eval()` switches layers that behave differently at train and test time (dropout, batch-norm) into their deterministic test behaviour — our current model has none, but it's an essential habit. `torch.no_grad()` tells PyTorch not to build the computation graph needed for backpropagation, making evaluation faster and lighter on memory since we will never call `backward()` here.

</details>

---

---
# Part 5 — Evaluation: beyond a single number

Accuracy is one number — useful, but it hides *what kind* of mistakes the model makes. A **confusion matrix** shows, for each true digit (rows), how often each digit was predicted (columns). A perfect model puts everything on the diagonal.

In [ ]:
confusion = torch.zeros(10, 10, dtype=torch.int64)

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        predictions = logits.argmax(dim=1).cpu()
        for true, pred in zip(labels, predictions):
            confusion[true, pred] += 1

fig, ax = plt.subplots(figsize=(6, 6))
ax.imshow(confusion, cmap="Blues")
ax.set_xticks(range(10))
ax.set_yticks(range(10))
ax.set_xlabel("predicted digit")
ax.set_ylabel("true digit")
ax.set_title("Confusion matrix (test set)")
for i in range(10):
    for j in range(10):
        colour = "white" if confusion[i, j] > confusion.max() // 2 else "black"
        ax.text(j, i, confusion[i, j].item(), ha="center", va="center", color=colour, fontsize=8)
plt.show()

Even more instructive is looking at the **individual mistakes**. Some are genuinely ambiguous scrawls that humans would also get wrong; others reveal systematic weaknesses of the model.

In [ ]:
wrong_images, wrong_true, wrong_pred = [], [], []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        predictions = logits.argmax(dim=1).cpu()
        mistakes = predictions != labels               # boolean mask over the batch
        wrong_images.append(images[mistakes])
        wrong_true.append(labels[mistakes])
        wrong_pred.append(predictions[mistakes])

wrong_images = torch.cat(wrong_images)
wrong_true = torch.cat(wrong_true)
wrong_pred = torch.cat(wrong_pred)
print(f"The model got {len(wrong_images)} of {len(test_dataset)} test images wrong.")

fig, axes = plt.subplots(2, 8, figsize=(12, 4))
for i, ax in enumerate(axes.flat):
    ax.axis("off")
    if i < len(wrong_images):
        ax.imshow(wrong_images[i, 0], cmap="gray")
        ax.set_title(f"true: {wrong_true[i].item()}  pred: {wrong_pred[i].item()}", fontsize=9)
plt.tight_layout()
plt.show()

---
### 🧠 Quick check 5

**Q:** Look at the off-diagonal entries of your confusion matrix. Which pairs of digits does the model confuse most, and can you guess why?

<details><summary><b>Show answer</b></summary>

The exact numbers vary run to run, but the usual suspects are **4 ↔ 9**, **3 ↔ 5**, **7 ↔ 2** and **8 ↔ 3** — pairs that share most of their pen strokes and differ in small details (whether the top of the 4/9 is closed, the top-left curve of 3/5, ...). The model's mistakes mirror the visual similarity structure of the digits themselves — sloppy handwriting sits genuinely between two classes.

</details>

---

---
# Part 6 — Exercises ✏️

Now it is your turn. Everything you need has appeared above — and note that our helper `train_one_epoch(...)` takes the model, loss and optimiser as *arguments* (and `accuracy(...)` takes any model and loader), so you can reuse both for every experiment below.

Solutions are at the very end of the notebook, but **write your own attempt first** — that is where the learning happens.

*Short on time? Prioritise Exercise 1; in Exercise 2 the label-smoothing option is a one-line change; Exercise 3 makes a good take-home.*

## Exercise 1 — Build a deeper CNN

`SimpleCNN` has **two** convolutional layers. Build a `DeeperCNN` with **three** conv blocks:

- three blocks of *conv (3 × 3, padding 1) → ReLU → max-pool (2 × 2)*, with channels going **1 → 16 → 32 → 64**;
- then flatten, `Linear(? → 128)`, ReLU, and `Linear(128 → 10)`.

Your main job is to work out the `?` — the flattened feature size after the third block. Use the shape arithmetic from Part 2 and mind the rounding!

**Before you train it, make a prediction:** will `DeeperCNN` have *more* or *fewer* parameters than `SimpleCNN`'s 206,922? Deeper should mean bigger... shouldn't it?

<details><summary><b>Hint (the rule)</b></summary>

Each 3 × 3 conv with padding 1 keeps the spatial size; each 2 × 2 max-pool halves it, **rounding down**. Check carefully what 7 becomes after pooling!

</details>

<details><summary><b>Full answer (only if you're stuck)</b></summary>

28 → 14 → 7 → **3** (7 halves to 3, not 3.5 — the pool simply drops the leftover row and column), so the flattened size is $64 \times 3 \times 3 = 576$.

</details>

In [ ]:
class DeeperCNN(nn.Module):
    def __init__(self):
        super().__init__()
        # TODO: define three conv layers (channels 1->16, 16->32, 32->64,
        #       each kernel_size=3, padding=1), one pooling layer,
        #       and two linear layers (? -> 128 and 128 -> 10).
        raise NotImplementedError("define the layers, then delete this line")

    def forward(self, x):
        # TODO: three times (conv -> relu -> pool), then flatten, fc1 -> relu, fc2.
        raise NotImplementedError("write the forward pass, then delete this line")

In [ ]:
# This cell trains your DeeperCNN. It will just print a reminder until
# you have completed the TODOs above.
try:
    deeper_model = DeeperCNN().to(device)

    # Sanity check before training:
    out = deeper_model(torch.randn(2, 1, 28, 28, device=device))
    assert list(out.shape) == [2, 10], f"expected output [2, 10], got {list(out.shape)}"

    n_simple = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_deeper = sum(p.numel() for p in deeper_model.parameters() if p.requires_grad)
    print(f"SimpleCNN: {n_simple:,} parameters | DeeperCNN: {n_deeper:,} parameters\n")

    deeper_optimiser = torch.optim.Adam(deeper_model.parameters(), lr=1e-3)
    for epoch in range(2):
        print(f"Epoch {epoch + 1}/2")
        train_one_epoch(deeper_model, train_loader, criterion, deeper_optimiser)
        print(f"   test accuracy: {accuracy(deeper_model, test_loader):.1%}\n")

except NotImplementedError as todo:
    print(f"⚠️ Not finished yet — {todo}")

## Exercise 2 — Try a different loss function

The loss function defines *what the network actually optimises*, and PyTorch provides a whole catalogue of them in [`torch.nn`](https://docs.pytorch.org/docs/stable/nn.html#loss-functions). Pick **at least one** alternative, train a *fresh* `SimpleCNN` with it, and compare the final test accuracy with the ~98–99% we got from plain cross-entropy. Some suggestions, roughly in order of difficulty:

1. **`nn.CrossEntropyLoss(label_smoothing=0.1)`** — instead of asking for 100% confidence in the true class, the target becomes a blend of 90% one-hot label and 10% *uniform distribution over all ten classes* — so the true digit gets probability 0.91 and every other digit 0.01. A widely-used regularisation trick — one argument, big literature.
2. **`nn.NLLLoss`** — the *negative log-likelihood* loss. It expects **log-probabilities**, not logits, so you must apply `F.log_softmax(..., dim=1)` at the end of your model's forward pass. In fact, `CrossEntropyLoss` *is exactly* `log_softmax` + `NLLLoss` — a good way to prove that to yourself.
3. **`nn.MSELoss`** *(stretch)* — mean squared error is a *regression* loss; to use it for classification you need to turn each label into a **one-hot vector** (`F.one_hot(labels, num_classes=10).float()`) and compare it with the model's (softmaxed) outputs. It works — but notice *how much slower* it learns, and think about why it is a poor fit for classification.

⚠️ One caveat: loss *values* from different loss functions are not comparable with each other — compare models by **accuracy**.

In [ ]:
# TODO 1: pick a loss function from torch.nn (see the link above).
new_criterion = ...        # e.g. nn.CrossEntropyLoss(label_smoothing=0.1)

# TODO 2 (only if your chosen loss needs it): some losses expect the model to
# output something other than raw logits — you may need to modify the model.
loss_model = SimpleCNN().to(device)

if new_criterion is ...:
    print("⚠️ Choose a loss function above (TODO 1), then re-run this cell.")
else:
    loss_optimiser = torch.optim.Adam(loss_model.parameters(), lr=1e-3)
    losses = []
    for epoch in range(2):
        print(f"Epoch {epoch + 1}/2")
        losses += train_one_epoch(loss_model, train_loader, new_criterion, loss_optimiser)
        print(f"   test accuracy: {accuracy(loss_model, test_loader):.1%}\n")

## Exercise 3 (optional stretch) — Optimisers and learning rates

If you have time left, investigate the *optimiser* side of training. Train a fresh `SimpleCNN` for one epoch with each of these settings and compare the loss curves and accuracies:

- `torch.optim.SGD(model.parameters(), lr=0.01, momentum=0.9)` — the classic;
- `torch.optim.Adam(..., lr=1e-3)` — our default;
- `torch.optim.Adam(..., lr=1.0)` — a learning rate that is far too large;
- `torch.optim.Adam(..., lr=1e-6)` — far too small.

You should see the whole story of the learning rate in four curves: too big *diverges or oscillates*, too small *crawls*, just right *converges quickly*.

In [ ]:
# Free experimentation cell — copy the training pattern from Exercise 2.
# Tip: collect each run's `losses` list under a different name, then plot them
# on the same axes to compare.

---
# Wrap-up

In one hour you have covered the complete deep learning workflow, end to end:

| Ingredient | What we used |
|---|---|
| **Data** | MNIST via `torchvision`, transforms, `DataLoader` batching |
| **Model** | a CNN: stacked *conv → ReLU → pool* blocks + linear classifier head |
| **Loss** | cross-entropy (and friends, in Exercise 2) |
| **Optimiser** | Adam / SGD from `torch.optim` |
| **Loop** | forward → loss → backward → step, over mini-batches |
| **Evaluation** | held-out test set, accuracy, confusion matrix, error inspection |

Everything larger — from ResNets to the vision models behind self-driving cars — is this same recipe with bigger ingredients.

**What we deliberately left out** (your next topics to explore): validation splits and hyperparameter tuning, data augmentation, batch normalisation and dropout, learning-rate schedules, modern architectures (ResNets, vision transformers), and transfer learning from pre-trained models.

### Where to go next

- [dl-pytorch](https://github.com/atapour/dl-pytorch) — Amir Atapour-Abarghouei's teaching examples (GANs, VAEs, transformers and more), which inspired this notebook
- [Official PyTorch tutorials](https://docs.pytorch.org/tutorials/) — start with the 60-minute blitz
- [3Blue1Brown's neural network series](https://www.3blue1brown.com/topics/neural-networks) — beautiful visual intuition for backpropagation
- [CS231n](https://cs231n.github.io/) — Stanford's convolutional networks course notes
- [BMVA](https://www.bmva.org/) — the British Machine Vision Association

---

# Solutions

Try the exercises first! Then compare with the reference implementations below.

<details><summary><b>Solution — Exercise 1: DeeperCNN</b></summary>

```python
class DeeperCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(64 * 3 * 3, 128)   # 28 -> 14 -> 7 -> 3, so 64*3*3 = 576
        self.fc2   = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))   # [B, 16, 14, 14]
        x = self.pool(F.relu(self.conv2(x)))   # [B, 32, 7, 7]
        x = self.pool(F.relu(self.conv3(x)))   # [B, 64, 3, 3]
        x = torch.flatten(x, start_dim=1)      # [B, 576]
        x = F.relu(self.fc1(x))
        return self.fc2(x)
```

**The parameter surprise:** DeeperCNN has only **98,442** parameters — *less than half* of SimpleCNN's 206,922, despite being deeper! The third conv block costs a modest 18,496 parameters but shrinks the feature map from 7 × 7 to 3 × 3, cutting `fc1` from $1568 \times 128$ down to $576 \times 128$ weights. This is a recurring theme in CNN design: **depth is cheap; big fully-connected layers are expensive.** Accuracy after 2 epochs is typically ~99%, similar to or slightly better than SimpleCNN.

</details>

<details><summary><b>Solution — Exercise 2: different loss functions</b></summary>

**Option 1 — label smoothing** is a one-line change:

```python
new_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
loss_model = SimpleCNN().to(device)
```

Accuracy stays in the ~98–99% range. Note the loss no longer approaches 0 — with smoothed targets, even a perfect model has non-zero loss, so its *value* is not comparable to the plain cross-entropy runs.

**Option 2 — NLLLoss** needs the model to output log-probabilities. The tidy way is a subclass that reuses `SimpleCNN`:

```python
class SimpleCNNLogSoftmax(SimpleCNN):
    def forward(self, x):
        return F.log_softmax(super().forward(x), dim=1)

new_criterion = nn.NLLLoss()
loss_model = SimpleCNNLogSoftmax().to(device)
```

Training behaves *identically* to `CrossEntropyLoss` — because it is the same function, just split into two steps. You can verify the equivalence directly:

```python
logits = torch.randn(4, 10)
labels = torch.tensor([1, 0, 3, 9])
print(nn.CrossEntropyLoss()(logits, labels))
print(nn.NLLLoss()(F.log_softmax(logits, dim=1), labels))   # same number
```

(And `accuracy` still works unchanged: `log_softmax` is monotonic, so the `argmax` is the same.)

**Option 3 — MSE on one-hot targets.** `train_one_epoch` calls `criterion(logits, labels)` with integer labels, so wrap the conversion inside a custom criterion function — a plain Python function works fine in place of an `nn` module:

```python
def mse_criterion(logits, labels):
    targets = F.one_hot(labels, num_classes=10).float()
    return F.mse_loss(torch.softmax(logits, dim=1), targets)

new_criterion = mse_criterion
loss_model = SimpleCNN().to(device)
```

It reaches a decent accuracy but learns visibly more slowly. MSE penalises a confident wrong answer only quadratically (the error per output is at most 1), whereas cross-entropy's $-\log p$ penalty grows *without bound* as the probability of the true class goes to 0 — giving much stronger gradients exactly where the model is most wrong. That, plus its clean probabilistic interpretation, is why cross-entropy is the default for classification.

</details>

<details><summary><b>Solution — Exercise 3: optimisers and learning rates</b></summary>

```python
runs = {
    "SGD, lr=0.01, momentum": lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9),
    "Adam, lr=1e-3":          lambda p: torch.optim.Adam(p, lr=1e-3),
    "Adam, lr=1.0":           lambda p: torch.optim.Adam(p, lr=1.0),
    "Adam, lr=1e-6":          lambda p: torch.optim.Adam(p, lr=1e-6),
}

plt.figure(figsize=(9, 4))
for name, make_optimiser in runs.items():
    torch.manual_seed(42)                      # same starting weights for a fair comparison
    run_model = SimpleCNN().to(device)
    run_losses = train_one_epoch(run_model, train_loader, criterion,
                                 make_optimiser(run_model.parameters()))
    smoothed = np.convolve(run_losses, np.ones(25) / 25, mode="valid")
    plt.plot(smoothed, label=f"{name}  (acc {accuracy(run_model, test_loader):.1%})")

plt.xlabel("training step")
plt.ylabel("loss (moving average)")
plt.yscale("log")
plt.legend()
plt.show()
```

Typical outcome: Adam at `1e-3` and SGD with momentum both converge nicely; Adam at `1.0` jumps around a large loss and never really learns; Adam at `1e-6` decreases *painfully* slowly. The learning rate is the first thing to tune when training misbehaves.

</details>

---

*Created for the BMVA Summer School. Inspired by, and gratefully acknowledging, Amir Atapour-Abarghouei's [dl-pytorch](https://github.com/atapour/dl-pytorch) materials.*